# 05 – Capacity Agent (Jira Tickets)

The **CapacityAgent** interacts with Jira: searching open incidents and creating new tickets.  
Mock mode uses an in-memory store — no Jira credentials needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.capacity_agent import CapacityAgent
from core.base_agent import AgentRequest

agent = CapacityAgent()

## 1. List open Jira tickets

In [ ]:
req = AgentRequest(query='Show open Jira bugs for retention', data_products=['retention'])
result = agent.execute(req)

print('Success  :', result.success)
print('Message  :', result.message)
print('Metadata :', result.metadata)

print('\nTickets:')
for t in result.data.get('tickets', []):
    print(f"  [{t['id']}] {t['summary']}")
    print(f"    status={t['status']}  priority={t['priority']}  type={t['issue_type']}")

## 2. Create a ticket manually

In [ ]:
req_create = AgentRequest(
    query='Create ticket: GRR dropped to 72% in EU region — pipeline failure',
    data_products=['retention'],
)
result = agent.execute(req_create)

print('Success  :', result.success)
print('Message  :', result.message)
print('Ticket ID:', result.data.get('ticket_id'))
print('Full ticket:')
ticket = result.data.get('ticket', {})
fields = ticket.get('fields', {})
print(f"  summary : {fields.get('summary')}")
print(f"  priority: {fields.get('priority', {}).get('name')}")
print(f"  labels  : {fields.get('labels')}")
print(f"  status  : {fields.get('status', {}).get('name')}")

## 3. Auto-ticket from anomaly (used by HITL node)

In [ ]:
anomaly = 'retention: GRR 78.0% is below threshold 85.0% — risk of missing targets'
result = agent.create_ticket_from_anomaly(
    anomaly_description=anomaly,
    product='retention',
    priority='High',
)

print('Ticket created:', result.data.get('ticket_id'))
print('Message       :', result.message)

## 4. Create multiple tickets for multiple products

In [ ]:
from services.jira.mock import MockJiraService

svc = MockJiraService()
fresh_agent = CapacityAgent(ticket_service=svc)

anomalies = [
    ('retention: GRR 78% below threshold 85%', 'retention', 'High'),
    ('cac: payback 42 months exceeds 36-month limit', 'cac', 'Medium'),
    ('ltv: LTV:CAC ratio 2.1 is below 3x minimum', 'ltv', 'High'),
]

for anomaly, product, priority in anomalies:
    r = fresh_agent.create_ticket_from_anomaly(anomaly, product=product, priority=priority)
    print(f'Created {r.data["ticket_id"]} [{priority}] for {product}')

print(f'\nTotal tickets in mock store: {len(svc.tickets)}')

## 5. Health check

In [ ]:
health = agent.health_check()
print(health)

## 6. Inspect canned Jira issues

In [ ]:
from services.jira.mock import _CANNED_ISSUES

print('Canned issues in mock Jira:')
for issue in _CANNED_ISSUES:
    f = issue['fields']
    print(f"  {issue['key']}  [{f['priority']['name']}]  {f['summary']}  ({f['status']['name']})")